# Competition-ready hybrid feasibility-first optimization

This notebook combines a dependency-free randomized-tree structural ensemble, the supplied L/D model, differential evolution, and local constrained Bayesian refinement. Its three phases deliberately treat stress as a hard cliff:

1. discover a diverse pool with a conservative stress UCB below **300 MPa**;
2. use differential evolution to minimize the official loss while retaining that margin;
3. use a local RFF Bayesian surrogate to refine the best regions while enforcing the final **335 MPa** stress UCB limit.

There is no callable nTop/FE solver in this repository. Final validation therefore re-runs the supplied L/D model and both surrogate ensembles after discrete-variable repair; it is explicitly surrogate-only validation.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from scipy.special import ndtr
from scipy.stats import qmc

ROOT = Path.cwd()
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'models' / 'ld_surrogate'))
from predict_ld import predict_ld_batch

RNG_SEED = 20260819
STRESS_PHASE1 = 300.0
STRESS_FINAL = 335.0
TOP_K = 10

TEST_CASES = [
    {'mission': 'High Speed Dash', 'altitude': 15.0, 'kcas': 250.0, 'aoa': 6.0, 'ld_target': 9.0, 'mass_target': 90.0, 'payload_target': 0.45, 'fuel_target': 0.15},
    {'mission': 'Max Endurance', 'altitude': 25.0, 'kcas': 150.0, 'aoa': 8.0, 'ld_target': 12.0, 'mass_target': 100.0, 'payload_target': 0.40, 'fuel_target': 0.18},
    {'mission': 'Max Capacity', 'altitude': 10.0, 'kcas': 180.0, 'aoa': 5.0, 'ld_target': 10.0, 'mass_target': 120.0, 'payload_target': 0.60, 'fuel_target': 0.20},
]

DESIGN_COLUMNS = ['C2/C1', 'C3/C1', 'C4/C1', 'B1/C1', 'B2/C1', 'B3/C1', 'X3/C1', 'S1', 'S3', 'C1', 'Skin Thickness', 'Front Spar Chord %', 'Rear Spar Chord %', 'Spar Thickness', '# of Ribs', 'Rib Thickness', 'Wingbox Cutout', '# of Fuselage Ribs', '# of Fuselage Spars', 'Fuselage Struct Thickness', 'Fuselage Struct Width']
TARGET_COLUMNS = ['Aircraft Empty Weight', 'Payload Volume', 'Fuel Volume', 'Max Hotspot Stress']
TARGET_LABELS = ['mass_kg', 'payload_m3', 'fuel_m3', 'stress_mpa']
BOUNDS = np.array([[.55,.85],[.18,.28],[.06,.09],[.1,.2],[.05,.2],[.35,.7],[.5,.65],[40,60],[20,40],[2500,4000],[.0003,.005],[.18,.35],[.55,.75],[.00098,.008],[3,14],[.0015,.015],[.01,.05],[0,4],[3,12],[.002,.025],[.001,.015]], float)
PHYSICAL_BOUNDS = BOUNDS.copy(); PHYSICAL_BOUNDS[17] = [3, 11]

df = pd.read_csv(ROOT / 'data' / 'bwb_structures_dataset.csv')
df = df.loc[df['Max Hotspot Stress'] < 1e4].reset_index(drop=True)
X = df[DESIGN_COLUMNS].to_numpy(float)
Y = df[TARGET_COLUMNS].to_numpy(float)
Y[:, 1:3] /= 1e9  # mm^3 to m^3
Y_LOG = np.log(np.maximum(Y, 1e-12))
rng = np.random.default_rng(RNG_SEED)
feasible_label = Y[:, 3] <= STRESS_FINAL
train_idx, test_idx = [], []
for label in (False, True):
    idx = np.flatnonzero(feasible_label == label); rng.shuffle(idx); split = int(.8 * len(idx)); train_idx.extend(idx[:split]); test_idx.extend(idx[split:])
train_idx, test_idx = np.array(train_idx), np.array(test_idx)
X_LO, X_HI = BOUNDS[:,0], BOUNDS[:,1]
def normalize(x): return (np.asarray(x) - X_LO) / (X_HI - X_LO)
def denormalize(x): return np.asarray(x) * (X_HI - X_LO) + X_LO

def repair_design(x):
    z = np.clip(np.asarray(x, float).copy(), X_LO, X_HI)
    z[..., 14] = np.rint(z[..., 14])
    z[..., 17] = 3 + 2 * np.rint(np.clip(z[..., 17], 0, 4))
    z[..., 18] = np.rint(z[..., 18])
    return z

print(f'Using {len(df):,} rows after artifact filtering; train={len(train_idx):,}, holdout={len(test_idx):,}.')

Using 13,597 rows after artifact filtering; train=10,877, holdout=2,720.


In [2]:
class ExtraRandomTreeEnsemble:
    """Pure NumPy extremely-randomized partition trees for multi-output regression."""
    def __init__(self, n_trees=28, max_depth=8, min_leaf=70, seed=RNG_SEED):
        self.n_trees, self.max_depth, self.min_leaf, self.seed = n_trees, max_depth, min_leaf, seed

    def fit(self, x, y):
        x, y = normalize(x), np.asarray(y)
        self.trees = []
        for tree_id in range(self.n_trees):
            local = np.random.default_rng(self.seed + tree_id)
            boot = local.integers(0, len(x), len(x))
            nodes = []
            def grow(rows, depth):
                value = y[rows].mean(axis=0)
                node_id = len(nodes); nodes.append([ -1, 0., -1, -1, value, len(rows) ])
                if depth >= self.max_depth or len(rows) < 2 * self.min_leaf:
                    return node_id
                features = local.choice(x.shape[1], size=min(7, x.shape[1]), replace=False)
                best = None
                parent = np.var(y[rows], axis=0).sum()
                for feature in features:
                    lo, hi = x[rows, feature].min(), x[rows, feature].max()
                    if hi <= lo: continue
                    cut = local.uniform(lo, hi); left = rows[x[rows, feature] <= cut]; right = rows[x[rows, feature] > cut]
                    if len(left) < self.min_leaf or len(right) < self.min_leaf: continue
                    score = parent - (len(left)*np.var(y[left],axis=0).sum() + len(right)*np.var(y[right],axis=0).sum()) / len(rows)
                    if best is None or score > best[0]: best = (score, feature, cut, left, right)
                if best is not None:
                    _, feature, cut, left, right = best
                    li, ri = grow(left, depth + 1), grow(right, depth + 1)
                    nodes[node_id][:4] = [feature, cut, li, ri]
                return node_id
            grow(boot, 0); self.trees.append(nodes)
        return self

    def member_predict(self, x):
        x = normalize(x); x = np.atleast_2d(x); out = np.empty((self.n_trees, len(x), 4))
        for t, nodes in enumerate(self.trees):
            for r, row in enumerate(x):
                node = 0
                while nodes[node][0] >= 0:
                    f, cut, li, ri = nodes[node][:4]; node = li if row[f] <= cut else ri
                out[t, r] = nodes[node][4]
        return out

    def predict(self, x):
        members = self.member_predict(x)
        return members.mean(axis=0), members.std(axis=0, ddof=1)

class RFFRidgeEnsemble:
    """Local BO model: bootstrapped random Fourier ridge regressors in log-output space."""
    def __init__(self, n_models=7, n_features=128, seed=RNG_SEED): self.n_models, self.n_features, self.seed = n_models, n_features, seed
    def fit(self, x, y):
        x = normalize(x); self.models = []
        for i in range(self.n_models):
            local = np.random.default_rng(self.seed + 1000 + i); rows = local.integers(0, len(x), len(x)); scale = local.uniform(1.3, 2.8)
            omega = local.normal(0, scale, (x.shape[1], self.n_features)); phase = local.uniform(0, 2*np.pi, self.n_features)
            phi = np.sqrt(2/self.n_features) * np.cos(x[rows] @ omega + phase); phi = np.c_[np.ones(len(phi)), phi]
            ridge = 10 ** local.uniform(-2.6, -1.3); weights = np.linalg.solve(phi.T @ phi + ridge*np.eye(phi.shape[1]), phi.T @ y[rows])
            self.models.append((omega, phase, weights))
        return self
    def predict(self, x):
        x = normalize(x); preds = []
        for omega, phase, weights in self.models:
            phi = np.sqrt(2/self.n_features) * np.cos(x @ omega + phase); preds.append(np.c_[np.ones(len(phi)), phi] @ weights)
        p = np.asarray(preds); return np.exp(p.mean(axis=0)), np.exp(p.std(axis=0, ddof=1))

tree = ExtraRandomTreeEnsemble().fit(X[train_idx], Y_LOG[train_idx])
rff = RFFRidgeEnsemble().fit(X[train_idx], Y_LOG[train_idx])
test_log, test_log_std = tree.predict(X[test_idx]); test_pred = np.exp(test_log)
mae = np.abs(test_pred - Y[test_idx]).mean(axis=0)
print('Tree holdout MAE:', dict(zip(TARGET_LABELS, np.round(mae, 4))))
print('Conservative feasibility accuracy:', round(np.mean((np.exp(test_log[:,3]) + 2*np.exp(test_log_std[:,3]) <= STRESS_FINAL) == (Y[test_idx,3] <= STRESS_FINAL)), 3))

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, name, actual, predicted in zip(axes.flat, TARGET_LABELS, Y[test_idx].T, test_pred.T):
    ax.scatter(actual, predicted, s=4, alpha=.28); lim = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]; ax.plot(lim, lim, 'k--', lw=1); ax.set(title=name, xlabel='actual', ylabel='tree prediction')
fig.tight_layout(); fig.savefig(OUTPUTS / 'hybrid_feasibility_first_model_validation.png', dpi=160); plt.show()

Tree holdout MAE: {'mass_kg': np.float64(31.2064), 'payload_m3': np.float64(0.1267), 'fuel_m3': np.float64(0.0636), 'stress_mpa': np.float64(344.1698)}
Conservative feasibility accuracy: 0.841


/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_84052/2069885431.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUTS / 'hybrid_feasibility_first_model_validation.png', dpi=160); plt.show()


In [3]:
def structural_prediction(designs, model=tree):
    d = repair_design(np.atleast_2d(designs)); log_mu, log_sd = model.predict(d)
    return d, np.exp(log_mu), np.exp(log_sd)

def normalize_for_search(designs):
    """Map physical repaired designs back to the latent optimizer coordinate."""
    z = np.asarray(designs, float).copy()
    z[..., 17] = (z[..., 17] - 3.0) / 2.0
    return normalize(z)

def predict_candidate(designs, mission, model=tree):
    d, mu, sd = structural_prediction(designs, model)
    frame = pd.DataFrame(d, columns=DESIGN_COLUMNS)
    ld = predict_ld_batch(frame, alt_kft=mission['altitude'], kcas=mission['kcas'], aoa=mission['aoa'])
    loss = ((ld - mission['ld_target']) / mission['ld_target'])**2 + ((mu[:,0] - mission['mass_target']) / mission['mass_target'])**2 + ((mu[:,1] - mission['payload_target']) / mission['payload_target'])**2 + ((mu[:,2] - mission['fuel_target']) / mission['fuel_target'])**2
    return d, ld, mu, sd, loss

def stress_ucb(mu, sd): return mu[:,3] + 2.0 * sd[:,3]
def penalized_loss(x, mission, threshold):
    _, _, mu, sd, loss = predict_candidate(np.asarray(x)[None,:], mission)
    excess = max(0., stress_ucb(mu, sd)[0] - threshold)
    return float(loss[0] + 500.0 * (excess / 20.0)**2)

def phase1_seeds(mission, seed):
    # Dataset exploration first: broad, real sampled geometry and conservative stress margin.
    d, ld, mu, sd, loss = predict_candidate(X, mission)
    good = np.flatnonzero(stress_ucb(mu, sd) <= STRESS_PHASE1)
    if len(good) < 40: good = np.argsort(stress_ucb(mu, sd))[:max(40, len(X)//30)]
    rank = good[np.argsort(loss[good])]
    # Greedy spread keeps several distinct feasible basins for DE.
    selected = [rank[0]]
    while len(selected) < min(24, len(rank)):
        candidates = rank[:min(500, len(rank))]; dist = ((normalize(d[candidates])[:,None,:] - normalize(d[selected])[None,:,:])**2).sum(axis=2).min(axis=1); selected.append(candidates[np.argmax(dist)])
    return d[selected], {'phase1_count': len(good), 'phase1_best_loss': float(loss[rank[0]])}

def local_bounds(center, radius=.14):
    c = normalize_for_search(center); return list(zip(np.maximum(0, c-radius), np.minimum(1, c+radius)))

def optimize_mission(mission, seed):
    seeds, info = phase1_seeds(mission, seed)
    # Phase 2: global, robust DE. The union of seed-centered broad boxes protects the learned feasible region.
    de_candidates = []
    for i, center in enumerate(seeds[:4]):
        result = differential_evolution(lambda z: penalized_loss(denormalize(z), mission, STRESS_PHASE1), local_bounds(center, .28), seed=seed+i, popsize=5, maxiter=18, polish=True, updating='immediate', workers=1)
        de_candidates.append(repair_design(denormalize(result.x)))
    de_candidates = np.asarray(de_candidates)
    _, _, de_mu, de_sd, de_loss = predict_candidate(de_candidates, mission)
    # Phase 3: RFF constrained BO. EI is weighted by a normal feasibility probability.
    incumbent = float(np.min(de_loss[stress_ucb(de_mu, de_sd) <= STRESS_FINAL])) if np.any(stress_ucb(de_mu, de_sd) <= STRESS_FINAL) else float(np.min(de_loss))
    refined = []
    for i, center in enumerate(de_candidates[np.argsort(de_loss)[:3]]):
        def negative_acquisition(z):
            candidate = repair_design(denormalize(z))[None,:]
            _, ld, mu, sd, loss = predict_candidate(candidate, mission, rff)
            sigma_loss = max(.02, .10 * (np.linalg.norm(sd[0,:3] / np.maximum(mu[0,:3], 1e-9))))
            zscore = (incumbent - loss[0]) / sigma_loss
            ei = (incumbent-loss[0])*ndtr(zscore) + sigma_loss*np.exp(-.5*zscore*zscore)/np.sqrt(2*np.pi)
            pf = ndtr((STRESS_FINAL - mu[0,3]) / max(sd[0,3], 1e-6))
            return float(-ei * pf)
        result = differential_evolution(negative_acquisition, local_bounds(center, .12), seed=seed+100+i, popsize=5, maxiter=12, polish=True, updating='immediate', workers=1)
        refined.append(repair_design(denormalize(result.x)))
    # Fill a short-list from all stages, then select only conservative final candidates.
    pool = np.vstack([seeds, de_candidates, np.asarray(refined)])
    pool = np.unique(np.round(pool, 10), axis=0)
    d, ld, mu, sd, loss = predict_candidate(pool, mission)
    ucb = stress_ucb(mu, sd); feasible = ucb <= STRESS_FINAL
    order = np.lexsort((loss, ~feasible))
    chosen = order[:TOP_K] if feasible.sum() >= TOP_K else order[:TOP_K]
    info.update({'de_candidates': len(de_candidates), 'refined_candidates': len(refined), 'final_feasible': int(feasible.sum())})
    return d[chosen], ld[chosen], mu[chosen], sd[chosen], loss[chosen], ucb[chosen], info

def validate(design, mission):
    d, ld, mu, sd, loss = predict_candidate(design[None,:], mission)
    z = d[0]; in_bounds = bool(np.all(z >= PHYSICAL_BOUNDS[:,0]) and np.all(z <= PHYSICAL_BOUNDS[:,1]))
    discrete = bool(z[14].is_integer() and z[18].is_integer() and z[17].is_integer() and int(z[17]) % 2 == 1)
    return d[0], float(ld[0]), mu[0], sd[0], float(loss[0]), float(stress_ucb(mu, sd)[0]), in_bounds, discrete

records, diagnostics = [], []
for case_no, mission in enumerate(TEST_CASES, 1):
    designs, lds, mus, sds, losses, ucbs, info = optimize_mission(mission, RNG_SEED + case_no)
    diagnostics.append({'mission': mission['mission'], **info})
    for rank, design in enumerate(designs, 1):
        repaired, ld, mu, sd, loss, ucb, in_bounds, discrete = validate(design, mission)
        row = {'mission': mission['mission'], 'rank': rank, 'source': 'hybrid_pool', 'official_loss': loss, 'predicted_ld': ld, 'predicted_mass_kg': mu[0], 'predicted_payload_m3': mu[1], 'predicted_fuel_m3': mu[2], 'predicted_stress_mpa': mu[3], 'stress_uncertainty_mpa': sd[3], 'stress_ucb_mpa': ucb, 'conservative_feasible': ucb <= STRESS_FINAL, 'in_bounds': in_bounds, 'discrete_valid': discrete, 'surrogate_only_validation': True}
        row.update(dict(zip(DESIGN_COLUMNS, repaired))); records.append(row)
results = pd.DataFrame(records).sort_values(['mission', 'rank']).reset_index(drop=True)
best = results.loc[results.groupby('mission')['rank'].idxmin()].reset_index(drop=True)
results.to_csv(OUTPUTS / 'hybrid_feasibility_first_results.csv', index=False)
best.to_csv(OUTPUTS / 'hybrid_feasibility_first_best.csv', index=False)
print(pd.DataFrame(diagnostics).to_string(index=False))
print(best[['mission','official_loss','predicted_ld','predicted_mass_kg','predicted_stress_mpa','stress_ucb_mpa','conservative_feasible']].to_string(index=False))

        mission  phase1_count  phase1_best_loss  de_candidates  refined_candidates  final_feasible
High Speed Dash          9193          0.084608              4                   3              31
  Max Endurance          9193          0.004774              4                   3              31
   Max Capacity          9193          0.041055              4                   3              31
        mission  official_loss  predicted_ld  predicted_mass_kg  predicted_stress_mpa  stress_ucb_mpa  conservative_feasible
High Speed Dash       0.000270      9.000001          90.991431            204.572837      209.838585                   True
   Max Capacity       0.000527     10.000000         121.307464            220.041401      224.206232                   True
  Max Endurance       0.000887     11.999826         102.085206             73.820667       77.638563                   True


In [4]:
# Final audit and competition-output visualizations.
assert len(results) == len(TEST_CASES) * TOP_K
assert results.groupby('mission').size().eq(TOP_K).all()
assert results['in_bounds'].all() and results['discrete_valid'].all()
assert not results.duplicated(['mission'] + DESIGN_COLUMNS).any()
assert (results.loc[results['conservative_feasible'], 'stress_ucb_mpa'] <= STRESS_FINAL + 1e-9).all()
print(f'Validated {len(results)} repaired candidates; {results.conservative_feasible.sum()} meet the conservative stress limit.')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for mission, group in results.groupby('mission', sort=False):
    axes[0].scatter(group['predicted_stress_mpa'], group['official_loss'], s=35, label=mission)
    axes[1].plot(group['rank'], group['stress_ucb_mpa'], marker='o', label=mission)
axes[0].axvline(STRESS_PHASE1, color='tab:green', ls='--', label='phase-1 margin'); axes[0].axvline(STRESS_FINAL, color='tab:red', ls='--', label='final UCB limit'); axes[0].set(xlabel='predicted stress [MPa]', ylabel='official loss', title='Hybrid shortlist trade-off')
axes[1].axhline(STRESS_FINAL, color='tab:red', ls='--', label='final UCB limit'); axes[1].set(xlabel='candidate rank', ylabel='stress UCB [MPa]', title='Conservative final validation')
for ax in axes: ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUTS / 'hybrid_feasibility_first_candidates.png', dpi=160); plt.show()

diag = pd.DataFrame(diagnostics)
fig, ax = plt.subplots(figsize=(9, 4.5)); x = np.arange(len(diag)); width=.25
ax.bar(x-width, diag['phase1_count'], width, label='phase-1 feasible dataset designs'); ax.bar(x, diag['de_candidates'], width, label='DE candidates'); ax.bar(x+width, diag['final_feasible'], width, label='final UCB-feasible pool')
ax.set(xticks=x, xticklabels=diag['mission'], ylabel='count', title='Feasibility-first search progression'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUTS / 'hybrid_feasibility_first_progression.png', dpi=160); plt.show()

print('Wrote:', OUTPUTS / 'hybrid_feasibility_first_results.csv')
print('Wrote:', OUTPUTS / 'hybrid_feasibility_first_best.csv')

Validated 30 repaired candidates; 30 meet the conservative stress limit.
Wrote: /Users/vaarijbetala/Desktop/ntopwork/nTop---ASME-IDETC-CIE-Student-Hackathon/outputs/hybrid_feasibility_first_results.csv
Wrote: /Users/vaarijbetala/Desktop/ntopwork/nTop---ASME-IDETC-CIE-Student-Hackathon/outputs/hybrid_feasibility_first_best.csv


/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_84052/2030375185.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUTS / 'hybrid_feasibility_first_candidates.png', dpi=160); plt.show()
/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_84052/2030375185.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUTS / 'hybrid_feasibility_first_progression.png', dpi=160); plt.show()
